# PLFS Data Analysis Pipeline
## Production-Grade Implementation

**Version**: 1.0.0  
**Last Updated**: April 2026  
**Author**: PLFS Research Team

---

### Overview
This notebook implements a production-ready data pipeline for analyzing PLFS (Periodic Labour Force Survey) data with:

- ✅ Industry-standard coding practices
- ✅ Complete data validation
- ✅ Survey weight handling (NSO methodology)
- ✅ Error handling and logging
- ✅ Web API-ready outputs

### Quick Start
1. Place your PLFS data files in `./data/raw/`
2. Run all cells sequentially
3. Results will be saved in `./data/output/`

## 1. Environment Setup

In [ ]:
# Install required packages (run once)
!pip install pandas numpy matplotlib seaborn plotly pyyaml openpyxl pyarrow fastparquet

: 

In [ ]:
# Import libraries
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Import custom pipeline module
from plfs_data_pipeline import (
    PLFSConfig,
    PLFSDataProcessor,
    PLFSAnalytics,
    DataExporter,
    setup_logging,
)

# Configure display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Environment setup complete")

## 2. Configuration

In [ ]:
# Configure from project config.yaml (multi-year)
cfg_path = Path("./config.yaml")
cfg_yaml = yaml.safe_load(cfg_path.read_text())

paths_cfg = cfg_yaml["paths"]
files_cfg = cfg_yaml["files"]
analysis_cfg = cfg_yaml.get("analysis", {})
validation_cfg = cfg_yaml.get("validation", {})
rounds_cfg = cfg_yaml["plfs_extracted"]["rounds"]

base_extracted_path = Path(paths_cfg["plfs_extracted"])

# Build a base config object; per-round configs will override raw_data_path/file names
base_config = PLFSConfig(
    raw_data_path=Path(paths_cfg["raw_data"]),
    processed_data_path=Path(paths_cfg["processed_data"]),
    output_path=Path(paths_cfg["output"]),
    household_file=files_cfg["household"],
    person_file=files_cfg["person"],
    csv_delimiter=files_cfg.get("delimiter", "\t"),
    field_dictionary_household=Path(files_cfg["field_dictionary_household"]) if files_cfg.get("field_dictionary_household") else None,
    field_dictionary_person=Path(files_cfg["field_dictionary_person"]) if files_cfg.get("field_dictionary_person") else None,
    estimate_type=analysis_cfg.get("estimate_type", "annual"),
    quarters=analysis_cfg.get("quarters", [1, 2, 3, 4]),
    status_measure=analysis_cfg.get("status_measure", "ups"),
    headline_min_age=analysis_cfg.get("headline_min_age", 15),
    nso_unemployment_rate=validation_cfg.get("nso_unemployment_rate", 6.7),
    validation_tolerance=validation_cfg.get("tolerance", 0.5),
    validation_sector=validation_cfg.get("sector"),
    validation_sex=validation_cfg.get("sex"),
    validation_strict_mode=validation_cfg.get("strict_mode", False),
)

# Setup logging
log_file = base_config.output_path / f"plfs_analysis_{datetime.now():%Y%m%d_%H%M%S}.log"
logger = setup_logging(log_file)

print("✓ Configuration complete")
print(f"  Base extracted path: {base_extracted_path}")
print(f"  Total configured rounds: {len(rounds_cfg)}")
print(f"  Output path: {base_config.output_path}")
print(f"  Log file: {log_file}")

## 3. Data Loading & Processing

In [ ]:
# Execute pipeline for all configured rounds

def pick_round_files(round_entry):
    hh = round_entry.get("household_v1") or round_entry.get("household_fv") or round_entry.get("household")
    pp = round_entry.get("person_v1") or round_entry.get("person_fv") or round_entry.get("person")
    return hh, pp

round_outputs = {}
trend_rows = []
analytics = PLFSAnalytics(logger, headline_min_age=base_config.headline_min_age)

print("Starting multi-year data processing pipeline...\n")

for round_name in sorted(rounds_cfg.keys()):
    round_entry = rounds_cfg[round_name]
    hh_file, person_file = pick_round_files(round_entry)
    if not hh_file or not person_file:
        print(f"⚠️ Skipping {round_name}: missing household/person mapping")
        continue

    round_config = PLFSConfig(
        raw_data_path=base_extracted_path / round_name,
        processed_data_path=base_config.processed_data_path,
        output_path=base_config.output_path,
        household_file=hh_file,
        person_file=person_file,
        csv_delimiter=base_config.csv_delimiter,
        field_dictionary_household=base_config.field_dictionary_household,
        field_dictionary_person=base_config.field_dictionary_person,
        estimate_type=base_config.estimate_type,
        quarters=base_config.quarters,
        status_measure=base_config.status_measure,
        headline_min_age=base_config.headline_min_age,
        nso_unemployment_rate=base_config.nso_unemployment_rate,
        validation_tolerance=base_config.validation_tolerance,
        validation_sector=base_config.validation_sector,
        validation_sex=base_config.validation_sex,
        validation_strict_mode=base_config.validation_strict_mode,
    )

    processor = PLFSDataProcessor(round_config, logger)
    round_df = processor.process_data()
    round_outputs[round_name] = round_df

    trend_rows.append({
        "round": round_name,
        "records": len(round_df),
        "households": int(round_df["HHID"].nunique()),
        "unemployment_rate": float(analytics.calculate_unemployment_rate(round_df)),
        "lfpr": float(analytics.calculate_lfpr(round_df)),
        "wpr": float(analytics.calculate_wpr(round_df)),
    })
    print(f"✓ {round_name}: {len(round_df):,} records processed")

# Keep latest round dataframe for downstream single-round cells
latest_round = sorted(round_outputs.keys())[-1]
processed_df = round_outputs[latest_round]

multiyear_stats = pd.DataFrame(trend_rows).sort_values("round").reset_index(drop=True)

print("\n✓ Multi-year pipeline complete!")
print(f"  Rounds processed: {len(multiyear_stats)}")
print(f"  Latest round in processed_df: {latest_round}")
print(f"  Latest round records: {len(processed_df):,}")

In [ ]:
# Multi-year preview + latest round preview
print("Multi-year trend table:")
display(multiyear_stats)

print("\nLatest-round preview:")
display(processed_df.head())

print("\nLatest-round data info:")
processed_df.info()

## 4. Summary Statistics

In [ ]:
# Initialize analytics engine for latest round
analytics = PLFSAnalytics(logger, headline_min_age=base_config.headline_min_age)

# Generate comprehensive summary statistics for latest round
summary_stats = analytics.generate_summary_statistics(processed_df)

print("="*80)
print("SUMMARY STATISTICS")
print("="*80)

print("\n📈 ALL-YEAR TREND (National)")
display(multiyear_stats[["round", "unemployment_rate", "lfpr", "wpr"]])

# Latest round national level
print("\n📊 LATEST ROUND INDICATORS")
print(f"  Unemployment Rate: {summary_stats['national']['unemployment_rate']:.2f}%")
print(f"  LFPR: {summary_stats['national']['lfpr']:.2f}%")
print(f"  WPR: {summary_stats['national']['wpr']:.2f}%")

# By gender (latest round)
print("\n👥 LATEST ROUND - BY GENDER")
print("\nUnemployment Rate:")
print(summary_stats['by_gender']['unemployment'])

# By sector (latest round)
print("\n🏙️  LATEST ROUND - BY SECTOR (Rural/Urban)")
print("\nUnemployment Rate:")
print(summary_stats['by_sector']['unemployment'])

## 5. Visualization

In [ ]:
# Create visualization helper
def create_unemployment_charts(df, stats, trend_df):
    """Create publication-quality unemployment visualizations."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('PLFS Labor Market Indicators', fontsize=16, fontweight='bold')
    
    # Chart 1: All-year trend
    ax1 = axes[0, 0]
    t = trend_df.copy()
    ax1.plot(t['round'], t['unemployment_rate'], marker='o', color='#e74c3c', label='UR')
    ax1.plot(t['round'], t['lfpr'], marker='o', color='#3498db', label='LFPR')
    ax1.plot(t['round'], t['wpr'], marker='o', color='#2ecc71', label='WPR')
    ax1.set_ylabel('Percentage (%)')
    ax1.set_title('All-Year Trend (National)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.legend()
    
    # Chart 2: Latest-round gender comparison
    ax2 = axes[0, 1]
    gender_unemp = stats['by_gender']['unemployment']
    ax2.bar(['Male', 'Female'], gender_unemp.values, color=['#3498db', '#e74c3c'])
    ax2.set_ylabel('Unemployment Rate (%)')
    ax2.set_title('Latest Round: Unemployment by Gender')
    for i, v in enumerate(gender_unemp.values):
        ax2.text(i, v + 0.3, f"{v:.1f}%", ha='center', fontweight='bold')
    
    # Chart 3: Latest-round rural-urban comparison
    ax3 = axes[1, 0]
    sector_unemp = stats['by_sector']['unemployment']
    ax3.bar(['Rural', 'Urban'], sector_unemp.values, color=['#27ae60', '#f39c12'])
    ax3.set_ylabel('Unemployment Rate (%)')
    ax3.set_title('Latest Round: Unemployment by Sector')
    for i, v in enumerate(sector_unemp.values):
        ax3.text(i, v + 0.3, f"{v:.1f}%", ha='center', fontweight='bold')
    
    # Chart 4: Latest-round top states
    ax4 = axes[1, 1]
    if 'by_state' in stats:
        state_unemp = stats['by_state']['unemployment'].sort_values(ascending=False).head(10)
        ax4.barh(range(len(state_unemp)), state_unemp.values, color='#9b59b6')
        ax4.set_yticks(range(len(state_unemp)))
        ax4.set_yticklabels([f"State {s}" for s in state_unemp.index])
        ax4.set_xlabel('Unemployment Rate (%)')
        ax4.set_title('Latest Round: Top 10 States by Unemployment')
        ax4.invert_yaxis()
    
    plt.tight_layout()
    
    # Save figure
    output_file = base_config.output_path / "unemployment_dashboard.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"✓ Saved visualization: {output_file}")
    
    return fig

# Generate charts
fig = create_unemployment_charts(processed_df, summary_stats, multiyear_stats)
plt.show()

## 6. Advanced Analytics

In [ ]:
# Age-group analysis (latest round)
print("📈 UNEMPLOYMENT BY AGE GROUP\n")

# Create age groups
processed_df['age_group'] = pd.cut(
    processed_df['AGE'],
    bins=[0, 14, 24, 34, 44, 54, 64, 120],
    labels=['0-14', '15-24', '25-34', '35-44', '45-54', '55-64', '65+']
)

# Calculate unemployment by age group
age_group_unemp = analytics.calculate_unemployment_rate(processed_df, by='age_group')

# Display results
print(age_group_unemp.sort_values(ascending=False))

# Visualize
plt.figure(figsize=(12, 6))
age_group_unemp.plot(kind='bar', color='steelblue')
plt.title('Latest Round: Unemployment Rate by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Unemployment Rate (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(base_config.output_path / "unemployment_by_age.png", dpi=300)
plt.show()

In [ ]:
# Education analysis (latest round)
print("🎓 UNEMPLOYMENT BY EDUCATION LEVEL\n")

if 'EDUCATION_LEVEL' in processed_df.columns:
    edu_unemp = analytics.calculate_unemployment_rate(processed_df, by='EDUCATION_LEVEL')
    print(edu_unemp.sort_values(ascending=False))
    
    # Visualize
    plt.figure(figsize=(12, 6))
    edu_unemp.sort_values().plot(kind='barh', color='coral')
    plt.title('Latest Round: Unemployment by Education Level', fontsize=14, fontweight='bold')
    plt.xlabel('Unemployment Rate (%)')
    plt.ylabel('Education Level')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(base_config.output_path / "unemployment_by_education.png", dpi=300)
    plt.show()
else:
    print("⚠️  Education level data not available")

## 7. Export Results for Web Application

In [ ]:
# Initialize exporter
exporter = DataExporter(base_config.output_path, logger)

# Export processed data (for further analysis)
print("Exporting data files...\n")

# 1. Latest-round processed dataset
exporter.export_to_parquet(
    processed_df,
    "plfs_processed_full.parquet"
)

# 2. All-year trend table
exporter.export_to_csv(
    multiyear_stats,
    "plfs_multiyear_trends.csv"
)

# 3. Latest-round summary statistics
exporter.export_summary_to_json(
    summary_stats,
    "summary_statistics.json"
)

# 4. Latest-round state-level data (for visualization)
if 'by_state' in summary_stats:
    state_df = pd.DataFrame({
        'state_code': summary_stats['by_state']['unemployment'].index,
        'unemployment_rate': summary_stats['by_state']['unemployment'].values
    })
    exporter.export_to_csv(state_df, "state_unemployment.csv")

# 5. Latest-round age-group analysis
age_group_df = pd.DataFrame({
    'age_group': age_group_unemp.index,
    'unemployment_rate': age_group_unemp.values
})
exporter.export_to_csv(age_group_df, "age_group_unemployment.csv")

print("\n✓ All exports complete!")
print(f"\nFiles saved to: {base_config.output_path}")

## 8. Generate Dashboard Data (for Website)

In [ ]:
# Create comprehensive dashboard data for web application
import json

dashboard_data = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "data_source": "PLFS multi-year",
        "total_rounds": int(len(multiyear_stats)),
        "latest_round": latest_round,
        "total_records_latest_round": int(len(processed_df)),
        "total_households_latest_round": int(processed_df['HHID'].nunique())
    },
    "national_indicators_latest_round": {
        "unemployment_rate": round(summary_stats['national']['unemployment_rate'], 2),
        "lfpr": round(summary_stats['national']['lfpr'], 2),
        "wpr": round(summary_stats['national']['wpr'], 2)
    },
    "multiyear_trend": multiyear_stats.to_dict(orient='records'),
    "demographics_latest_round": {
        "by_gender": {
            "male": {
                "unemployment": round(summary_stats['by_gender']['unemployment'][1], 2),
                "lfpr": round(summary_stats['by_gender']['lfpr'][1], 2)
            },
            "female": {
                "unemployment": round(summary_stats['by_gender']['unemployment'][2], 2),
                "lfpr": round(summary_stats['by_gender']['lfpr'][2], 2)
            }
        },
        "by_sector": {
            "rural": {
                "unemployment": round(summary_stats['by_sector']['unemployment'][1], 2)
            },
            "urban": {
                "unemployment": round(summary_stats['by_sector']['unemployment'][2], 2)
            }
        }
    },
    "age_groups_latest_round": [
        {
            "group": str(group),
            "unemployment_rate": round(rate, 2)
        }
        for group, rate in age_group_unemp.items()
    ]
}

# Save dashboard data
dashboard_file = base_config.output_path / "dashboard_data.json"
with open(dashboard_file, 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print(f"✓ Dashboard data exported to: {dashboard_file}")
print("\nPreview:")
print(json.dumps(dashboard_data, indent=2)[:500] + "...")

## 9. Validation Report

In [ ]:
# Generate validation report
print("="*80)
print("VALIDATION REPORT")
print("="*80)

# Latest-round benchmark validation
estimated_unemp = summary_stats['national']['unemployment_rate']
nso_official = base_config.nso_unemployment_rate
difference = abs(estimated_unemp - nso_official)

print("\n📊 LATEST-ROUND WEIGHT VALIDATION")
print(f"  Round: {latest_round}")
print(f"  Your Estimate: {estimated_unemp:.2f}%")
print(f"  NSO Benchmark: {nso_official:.2f}%")
print(f"  Difference: {difference:.2f} pp")
print(f"  Tolerance: {base_config.validation_tolerance:.2f} pp")

if difference <= base_config.validation_tolerance:
    print("  Status: ✅ PASSED")
else:
    print("  Status: ⚠️ MISMATCH (check definition slice)")

# Multi-year quality checks
print("\n🔍 MULTI-YEAR CHECK")
print(f"  Rounds Processed: {len(multiyear_stats):,}")
print(f"  Total Persons Across Rounds: {int(multiyear_stats['records'].sum()):,}")
print(f"  UR Range Across Rounds: {multiyear_stats['unemployment_rate'].min():.2f} - {multiyear_stats['unemployment_rate'].max():.2f}")

# Latest-round data quality checks
print("\n🔍 LATEST-ROUND DATA QUALITY")
print(f"  Total Records: {len(processed_df):,}")
print(f"  Missing Values: {processed_df.isnull().sum().sum():,}")
print(f"  Duplicate HHIDs: {processed_df['HHID'].duplicated().sum():,}")
print(f"  Weight Range: {processed_df['WEIGHT'].min():.2f} - {processed_df['WEIGHT'].max():.2f}")

print("\n" + "="*80)

## 10. Next Steps & Usage Guide

### Output Files Generated

All files are saved in `./data/output/`:

1. **plfs_processed_full.parquet** - Full processed dataset (efficient format)
2. **summary_statistics.json** - Summary statistics for web API
3. **dashboard_data.json** - Dashboard-ready data for website
4. **state_unemployment.csv** - State-level unemployment rates
5. **age_group_unemployment.csv** - Age-group unemployment rates
6. **unemployment_dashboard.png** - Visualization dashboard
7. **plfs_analysis_YYYYMMDD_HHMMSS.log** - Complete execution log

### For Web Application

Use the JSON files for your API:

```python
# Flask/FastAPI example
import json

@app.route('/api/dashboard')
def get_dashboard_data():
    with open('data/output/dashboard_data.json') as f:
        return json.load(f)
```

### For Further Analysis

Load the processed data:

```python
# Load processed data for custom analysis
df = pd.read_parquet('data/output/plfs_processed_full.parquet')

# Run custom calculations
custom_stat = np.average(df['your_variable'], weights=df['WEIGHT'])
```

### Troubleshooting

If validation fails:
1. Check that MLTS column exists in raw data
2. Verify you're using correct year/quarters
3. Review the log file for detailed error messages
4. Ensure casualty cases are excluded (survey_code != 3)